In [186]:
pip install --upgrade --quiet google-cloud-aiplatform[agent_engines,adk] googlemaps requests

In [187]:
import os
import json
import requests
from typing import Tuple, Dict, Any, Optional, Sequence, Callable
import logging

# Import Google ADK and Vertex AI Reasoning Engine
import vertexai
from google.adk.agents import Agent
from vertexai.agent_engines import AdkApp

# For safety settings with Vertex AI GenerativeModel
from vertexai.preview.generative_models import GenerativeModel, HarmCategory, HarmBlockThreshold, SafetySetting, GenerationResponse

In [188]:
# --- Configuration ---
PROJECT_ID: str = os.getenv("GOOGLE_CLOUD_PROJECT_ID", "qwiklabs-gcp-04-70cfc463a7f7")
LOCATION: str = os.getenv("GOOGLE_CLOUD_REGION", "us-central1") # e.g., "us-central1"
GOOGLE_MAPS_API_KEY: str = os.getenv("GOOGLE_MAPS_API_KEY", "")

# User-Agent for NWS API requests
NWS_USER_AGENT: str = "MyWeatherApp/1.0"

# Initialize Vertex AI
vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Vertex AI Initialized for Project: {PROJECT_ID}, Location: {LOCATION}")

# --- Set up basic logging for callbacks ---
# Using basicConfig for consistent logging, but relying on print() for critical debug info in callbacks
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

Vertex AI Initialized for Project: qwiklabs-gcp-04-70cfc463a7f7, Location: us-central1


In [189]:
# --- Helper functions for callbacks ---

def _extract_location_from_prompt(prompt: str) -> Optional[str]:
    """
    A simple heuristic to extract a potential city name from a prompt.
    This is a very basic implementation and might not catch all cases.
    """
    prompt_lower = prompt.lower()
    keywords = ["weather in", "forecast for", "temperature in", "how's the weather in"]
    for keyword in keywords:
        if keyword in prompt_lower:
            # Try to get the part after the keyword
            parts = prompt_lower.split(keyword, 1)
            if len(parts) > 1:
                # Take the rest of the string and try to clean it
                location_candidate = parts[1].strip()
                # Remove punctuation or trailing questions
                location_candidate = location_candidate.split('?')[0].split('.')[0].strip()
                return location_candidate
    return None

def _is_location_in_united_states(place_name: str) -> bool:
    """
    Checks if a given place name corresponds to a location within the United States
    using Google Maps Geocoding API.
    """
    if not GOOGLE_MAPS_API_KEY or GOOGLE_MAPS_API_KEY == "YOUR_GOOGLE_MAPS_API_KEY":
        logger.warning("Google Maps API Key is not set. Cannot verify location for US check. Defaulting to True.")
        return True # Default to True if API key is missing to avoid blocking everything

    geocode_url: str = "https://maps.googleapis.com/maps/api/geocode/json"
    params: Dict[str, str] = {
        "address": place_name,
        "key": GOOGLE_MAPS_API_KEY
    }
    try:
        response = requests.get(geocode_url, params=params, timeout=5)
        response.raise_for_status()
        data: Dict[str, Any] = response.json()

        if data["status"] == "OK" and data["results"]:
            for component in data["results"][0]["address_components"]:
                if "country" in component["types"] and component["short_name"] == "US":
                    return True
        return False
    except requests.exceptions.RequestException as e:
        logger.error(f"Error during geocoding API request for US check: {e}. Defaulting to True.")
        return True # Default to True if API call fails
    except json.JSONDecodeError as e:
        logger.error(f"Error decoding geocoding API response for US check: {e}. Defaulting to True.")
        return True # Default to True if JSON decode fails

def _check_for_malicious_input(prompt: str) -> bool:
    """
    Checks for malicious input using Vertex AI Safety Settings.
    """

    # Initialize the Vertex AI Generative Model for safety checks
    safety_model = GenerativeModel("gemini-2.5-flash-lite")

    # Define safety settings. Adjust thresholds as needed.
    safety_settings: Sequence[SafetySetting] = [
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_HARASSMENT,
            threshold=HarmBlockThreshold.BLOCK_ONLY_HIGH
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=HarmBlockThreshold.BLOCK_ONLY_HIGH
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=HarmBlockThreshold.BLOCK_ONLY_HIGH
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
            threshold=HarmBlockThreshold.BLOCK_ONLY_HIGH
        ),
    ]

    try:
        response = safety_model.generate_content(
            prompt,
            safety_settings=safety_settings
        )

        if response.prompt_feedback and response.prompt_feedback.block_reason:
            logger.warning(
                f"Vertex AI Safety blocked prompt due to: "
                f"{response.prompt_feedback.block_reason}. "
                f"Safety ratings: {response.prompt_feedback.safety_ratings}"
            )
            return False # Input is malicious based on Vertex AI Safety
        else:
            logger.info("Vertex AI Safety check passed. Input is not malicious.")
            return True # Input passed Vertex AI Safety check
    except Exception as e:
        logger.error(f"Error during Vertex AI Safety check: {e}. Allowing input to proceed.")
        return True # Default to allowing if the safety check itself fails

In [190]:
# --- Callback Functions ---

def before_model_callback(query: str) -> str:
    """
    Callback function executed before the model processes the user's prompt.
    1. Logs the user prompt.
    2. Ensures the user's location (if specified) is in the United States.
    3. Checks for malicious input using Vertex AI Safety Settings.
    """
    # Removed direct print statements, now using logger.info
    logger.info(f"BEFORE_MODEL_CALLBACK - User Query: '{query}'")

    # 2. Ensure the user's location is in the United States
    location_candidate = _extract_location_from_prompt(query)
    if location_candidate:
        logger.info(f"BEFORE_MODEL_CALLBACK - Extracted location candidate: '{location_candidate}'")
        if not _is_location_in_united_states(location_candidate):
            error_message = f"Location '{location_candidate}' is not in the United States. This agent only provides weather for US locations."
            logger.error(error_message)
            raise ValueError(error_message)
        else:
            logger.info(f"BEFORE_MODEL_CALLBACK - Location '{location_candidate}' confirmed to be in the United States.")
    else:
        logger.info("BEFORE_MODEL_CALLBACK - No specific location found in query for US check. Proceeding.")

    # 3. Check for malicious input using Vertex AI Safety
    if not _check_for_malicious_input(query):
        error_message = "Malicious input detected by Vertex AI Safety. Request blocked."
        logger.error(error_message) # Using logger.error for blocked requests
        raise ValueError(error_message)

    return query # Return the (potentially unchanged) query for model processing

def after_model_callback(query: str, model_response_text: str) -> None:
    """
    Callback function executed after the model generates a response.
    1. Logs the model response.
    """
    # Removed direct print statements, now using logger.info
    logger.info(f"AFTER_MODEL_CALLBACK - Original Query: '{query}'")
    logger.info(f"AFTER_MODEL_CALLBACK - Model Response: '{model_response_text.strip()}'")

In [191]:
# --- Tool Definitions (Python functions) ---

def get_lat_long_from_place(place_name: str) -> Optional[Dict[str, float]]:
    """
    Converts a human-readable place name into its latitude and longitude coordinates
    using the Google Maps Geocoding API.

    Args:
        place_name: The name of the place (e.g., "New York City", "London").

    Returns:
        A dictionary containing 'latitude' and 'longitude' as floats, or None if
        geocoding fails.
    """
    if not GOOGLE_MAPS_API_KEY or GOOGLE_MAPS_API_KEY == "YOUR_GOOGLE_MAPS_API_KEY":
        print("Google Maps API Key is not set. Cannot perform geocoding.")
        return None

    geocode_url: str = "https://maps.googleapis.com/maps/api/geocode/json"
    params: Dict[str, str] = {
        "address": place_name,
        "key": GOOGLE_MAPS_API_KEY
    }
    try:
        response = requests.get(geocode_url, params=params, timeout=5)
        response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)
        data: Dict[str, Any] = response.json()

        if data["status"] == "OK" and data["results"]:
            location = data["results"][0]["geometry"]["location"]
            return {"latitude": location["lat"], "longitude": location["lng"]}
        else:
            print(f"Geocoding failed for '{place_name}': {data.get('error_message', data['status'])}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Error during geocoding API request: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"Error decoding geocoding API response: {e}")
        return None

def get_nws_weather_forecast(latitude: float, longitude: float) -> Optional[Dict[str, Any]]:
    """
    Retrieves the current weather forecast for a given latitude and longitude
    using the National Weather Service (NWS) API.

    This is a two-step process:
    1. Get the forecast office and grid points from the latitude/longitude.
    2. Use the forecast URL from the first step to get the detailed forecast.

    Args:
        latitude: The latitude coordinate.
        longitude: The longitude coordinate.

    Returns:
        A dictionary containing the weather forecast data, or None if the retrieval fails.
    """
    base_url: str = "https://api.weather.gov"
    points_url: str = f"{base_url}/points/{latitude},{longitude}"

    headers: Dict[str, str] = {"User-Agent": NWS_USER_AGENT}

    try:
        # Step 1: Get forecast office and grid points
        response_points = requests.get(points_url, headers=headers, timeout=5)
        response_points.raise_for_status()
        points_data: Dict[str, Any] = response_points.json()

        forecast_url: Optional[str] = points_data["properties"].get("forecast")
        if not forecast_url:
            print(f"Could not find forecast URL for {latitude},{longitude}.")
            return None

        # Step 2: Get the detailed forecast
        response_forecast = requests.get(forecast_url, headers=headers, timeout=5)
        response_forecast.raise_for_status()
        forecast_data: Dict[str, Any] = response_forecast.json()

        return forecast_data

    except requests.exceptions.RequestException as e:
        print(f"Error during NWS API request: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"Error decoding NWS API response: {e}")
        return None
    except KeyError as e:
        print(f"Missing key in NWS API response: {e}")
        return None

def get_weather_summary(place_name: str) -> str:
    """
    Provides a weather summary or alert for a given place name.
    This function internally uses get_lat_long_from_place and get_nws_weather_forecast.

    Args:
        place_name: The name of the location for which to get the weather.

    Returns:
        A string containing the weather summary or an error message.
    """
    location_data: Optional[Dict[str, float]] = get_lat_long_from_place(place_name)
    if not location_data:
        return f"Could not find coordinates for {place_name}. Please try again with a different location."

    latitude: float = location_data["latitude"]
    # FIX: Changed 'lng' to 'longitude' to match the key returned by get_lat_long_from_place
    longitude: float = location_data["longitude"]

    weather_data: Optional[Dict[str, Any]] = get_nws_weather_forecast(latitude, longitude)
    if not weather_data:
        return f"Could not retrieve weather data for {place_name}."

    properties: Dict[str, Any] = weather_data.get("properties", {})
    forecast_periods: Optional[list] = properties.get("periods")

    if forecast_periods:
        current_or_next_period: Dict[str, Any] = forecast_periods[0] # Get the current or next forecast period
        name: str = current_or_next_period.get("name", "N/A")
        temperature: str = str(current_or_next_period.get("temperature", "N/A"))
        temperature_unit: str = current_or_next_period.get("temperatureUnit", "")
        forecast_text: str = current_or_next_period.get("detailedForecast", current_or_next_period.get("shortForecast", "No detailed forecast available."))

        summary: str = (
            f"**Weather for {place_name} ({name} Period):**\n"
            f"- Temperature: {temperature}°{temperature_unit}\n"
            f"- Conditions: {forecast_text}\n"
        )
        return summary
    else:
        return f"No forecast periods found for {place_name}."

In [192]:
# --- Create the Agent ---

al_agent: Agent = Agent(
    model="gemini-2.0-flash", # Using a newer, recommended model version
    name="Al",
    description="A weather agent that provides real-time weather summaries for user-specified locations.",
    instruction=(
        "You are Al, a helpful AI assistant specialized in providing weather information. "
        "Your primary goal is to fetch and summarize real-time weather for any city in the USA provided by the user. "
        "Use your available tools to achieve this. "
        "If the user asks for weather, always try to use the 'get_weather_summary' tool. "
        "If the user says 'bye' or indicates they are finished, provide a polite closing response."
    ),
    tools=[
        get_lat_long_from_place,
        get_nws_weather_forecast,
        get_weather_summary,
    ],
    # Removed before_query_hooks and after_query_hooks as they are not valid Agent constructor arguments
)

# AdkApp now only needs the agent
weather_app: AdkApp = AdkApp(
    agent=al_agent,
)
print("Agent 'Al' and Weather App created with callbacks.")

Agent 'Al' and Weather App created with callbacks.


In [193]:
# --- Helper function to process agent events ---
def process_agent_event(event: Dict[str, Any]) -> str: # Modified to return text for after_model_callback
    """
    Processes a single event from the agent's async_stream_query, assuming
    it is a dictionary and extracting information from its 'content.parts' structure.
    Returns the concatenated text content.
    """
    content = event.get('content')
    if not content or not isinstance(content, dict):
        return ""

    parts = content.get('parts')
    if not parts or not isinstance(parts, list) or not parts:
        return ""

    text_content = []
    for part in parts:
        if not isinstance(part, dict):
            continue

        if 'text' in part:
            print(part['text']) # Original printing
            text_content.append(part['text'])

        # You might also want to print function calls/responses for debugging
        if 'function_call' in part:
            function_call_data = part['function_call']
            if isinstance(function_call_data, dict):
                function_name = function_call_data.get('name')
                function_args = function_call_data.get('args')
                # print(f"DEBUG: Agent called tool: {function_name} with args: {function_args}")

        if 'function_response' in part:
            function_response_data = part['function_response']
            if isinstance(function_response_data, dict) and 'response' in function_response_data:
                response_result = function_response_data['response'].get('result')
                # print(f"DEBUG: Tool response: {response_result}")
    return "".join(text_content)

In [194]:
# --- User Session and Interaction ---

async def run_weather_session() -> None:
    """
    Creates a user session and interacts with the weather agent Al.
    Processes input, displays agent responses in Markdown, and handles
    session termination, integrating before/after callbacks.
    """
    print("\n--- Starting Weather Agent Session ---")
    print("Type 'bye' or 'exit' to end the session.")

    user_id: str = "test_user_123"

    while True:
        user_input: str = input("\nYou: ")
        if user_input.lower() in ["bye", "exit", "quit", "goodbye"]:
            print("\nAl: Goodbye! Have a great day!")
            break

        print("Al:")
        processed_input = user_input # Initialize with original input

        try:
            # --- Invoke before_model_callback ---
            processed_input = before_model_callback(user_input)

            full_model_response_text = "" # To aggregate streaming response for after_model_callback
            async for event in weather_app.async_stream_query(
                user_id=user_id,
                message=processed_input, # Use the processed input
            ):
                if isinstance(event, dict):
                    # Process event and accumulate text response
                    full_model_response_text += process_agent_event(event)
                else:
                    print(f"ERROR: Received non-dictionary event: {event}. Cannot process.")

            # --- Invoke after_model_callback ---
            if full_model_response_text: # Only log if there was an actual response
                after_model_callback(user_input, full_model_response_text)

        except ValueError as e: # Catch ValueErrors raised by callbacks (e.g., safety, location)
            print(f"Al: An issue occurred with your request: {e}")
        except Exception as e:
            print(f"An error occurred during agent interaction: {e}")

# To run the async function in a Jupyter notebook
import asyncio
# Uncomment the line below to run the interactive session
# asyncio.run(run_weather_session())

In [195]:
# --- Test Code for Multiple Cities in the USA and callback demonstration ---

async def test_agent_multiple_cities() -> None:
    """
    Tests the 'Al' agent with a predefined list of US cities and prints
    their weather summaries, including tests for callback functionalities.
    """
    print("\n--- Testing Agent Al with Multiple US Cities and Callbacks ---")

    cities_to_test: list[str] = [
        "What is the weather like in New York City, NY?",
        "Tell me about the forecast for Los Angeles, CA.",
        "How's the weather in Chicago, IL?",
        "Weather in Houston, TX?",
        "What's the temperature in Denver, CO?",
        "What is the weather like in Paris, France?", # Test non-US location, should be blocked
        "Tell me about something dangerous.", # Test malicious input, should be blocked by Vertex AI Safety
        "What is the weather like?", # Test prompt without explicit location, should proceed
    ]

    for user_query in cities_to_test:
        print(f"\n--- Querying weather for: '{user_query}' ---")
        print(f"You: {user_query}")
        print("Al:")
        processed_input = user_query # Initialize

        try:
            # --- Invoke before_model_callback ---
            processed_input = before_model_callback(user_query)

            full_model_response_text = "" # To aggregate streaming response for after_model_callback
            async for event in weather_app.async_stream_query(
                user_id="test_runner",
                message=processed_input, # Use the processed input
            ):
                if isinstance(event, dict):
                    full_model_response_text += process_agent_event(event)
                else:
                    print(f"ERROR: Received non-dictionary event: {event}. Cannot process.")

            # --- Invoke after_model_callback ---
            if full_model_response_text: # Only log if there was an actual response
                after_model_callback(user_query, full_model_response_text)

        except ValueError as e: # Catch ValueErrors raised by callbacks
            print(f"Al: An issue occurred with your request: {e}")
        except Exception as e:
            print(f"An error occurred during agent interaction for '{user_query}': {e}")
        print("-" * 40)

# Run the test cases
await test_agent_multiple_cities()


--- Testing Agent Al with Multiple US Cities and Callbacks ---

--- Querying weather for: 'What is the weather like in New York City, NY?' ---
You: What is the weather like in New York City, NY?
Al:
OK. The weather in New York City, NY this afternoon will be cloudy with a high near 43°F. There is a 70% chance of rain, with northeast winds around 10 mph. New rainfall amounts will be less than a tenth of an inch possible.

----------------------------------------

--- Querying weather for: 'Tell me about the forecast for Los Angeles, CA.' ---
You: Tell me about the forecast for Los Angeles, CA.
Al:
OK. The weather for Los Angeles, CA today will be sunny with a high near 75 degrees. The wind will be from the north at 10 to 15 mph.

----------------------------------------

--- Querying weather for: 'How's the weather in Chicago, IL?' ---
You: How's the weather in Chicago, IL?
Al:
Areas of fog and a slight chance of rain showers. Cloudy with a high near 43°F in Chicago, IL. The wind will 